# 03 - plot_psd_waterfall\n
Derived from `notebooks/prior_versions/plume_path_plot_clean.ipynb`.\n

In [12]:
import os, platform, socket

def is_server() -> bool:
    # strong signals
    if os.getenv("JUPYTERHUB_API_URL") or os.getenv("JUPYTERHUB_USER"):
        return True
    if os.getenv("SLURM_JOB_ID"):
        return True

    # fallback heuristic (common: local macOS, server Linux)
    return platform.system() != "Darwin"

print("system:", platform.system(), platform.release())
print("hostname:", socket.gethostname())
print("is_server:", is_server())

system: Linux 4.18.0-553.104.1.el8_10.x86_64
hostname: levante4.lvt.dkrz.de
is_server: True


In [13]:
from __future__ import annotations

import re, sys
from pathlib import Path
import colormaps as pcmaps
import matplotlib.axes as maxes
import matplotlib.dates as md
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.colors import LogNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FixedLocator, LogLocator, FuncFormatter

xr.set_options(keep_attrs=True)

sys.path.insert(0, str((Path.cwd() / ".." / "src").resolve()))
from utilities import make_pastel, load_plume_path_runs
from utilities.holimo_helpers import load_and_prepare_holimo



In [14]:
def create_new_jet3(n_colors=256, n_ice_colors=32):
    ice_colors = pcmaps.ice(np.linspace(1, 0.5, n_ice_colors))
    bk_colors  = pcmaps.BkBlAqGrYeOrReViWh200(np.linspace(0.1, 0.9, n_colors))
    transition = np.linspace(ice_colors[-1], bk_colors[0], 16)[1:-1]
    return ListedColormap(np.vstack([ice_colors, transition, bk_colors]))

new_jet4 = create_new_jet3(1024)

new_jet3 = create_new_jet3()
new_jet3_soft = make_pastel(new_jet3, desaturation=0.25, darken=0.90)


def format_axis_ticks(
    self,
    *,
    y_tick_pos: str = "both",
    x_tick_pos: str = "bottom",
    y_tick_labels: bool = False,
    x_tick_labels: bool = False,
    y_label: str = "",
    x_label: str = "",
    grid: bool = True,
):
    self.tick_params(which="both", direction="out", top=True, right=True, bottom=True, left=True)
    self.minorticks_on()
    self.tick_params(which="major", direction="out", length=5, width=0.9)
    self.tick_params(which="minor", direction="out", length=3, width=0.5)
    self.set_xlabel(x_label)
    self.set_ylabel(y_label)
    self.yaxis.set_ticks_position(y_tick_pos)
    self.xaxis.set_ticks_position(x_tick_pos)
    if not y_tick_labels:
        self.yaxis.set_ticklabels([])
    else:
        self.yaxis.set_major_formatter(plt.FuncFormatter(
            lambda x, p: f'{x:.3f}'.rstrip('0').rstrip('.') if x < 1 else f'{x:.0f}'))
    if not x_tick_labels:
        self.xaxis.set_ticklabels([])
    if grid:
        self.grid(True, which="major", linestyle="--", linewidth=0.25, color="black", alpha=0.6)
        self.grid(True, which="minor", linestyle=":", linewidth=0.15, color="black", alpha=0.35)
    self.set_axisbelow(False)


def format_elapsed_time(
    self,
    t0,
    t_end,
    *,
    major_interval: int = 5,
    minor_interval: int = 1,
    max_major_ticks: int = 14,
    max_minor_ticks: int = 180,
):
    """Format elapsed-time axis in minutes (numeric x axis)."""
    duration_min = float((t_end - t0) / np.timedelta64(1, "m"))
    if not np.isfinite(duration_min) or duration_min <= 0:
        return

    major_step = max(int(major_interval), int(np.ceil(duration_min / max_major_ticks)))
    minor_step = max(int(minor_interval), int(np.ceil(duration_min / max_minor_ticks)))

    major_times = np.arange(0, duration_min + major_step, major_step)
    minor_times = np.arange(0, duration_min + minor_step, minor_step)

    self.xaxis.set_major_locator(FixedLocator(major_times))
    self.xaxis.set_minor_locator(FixedLocator(minor_times))
    self.set_xticklabels([f"+{int(t):02d}" for t in major_times])


def add_missing_data_patches(
    self,
    da: xr.DataArray,
    *,
    min_consecutive: int = 2,
    add_legend: bool = True,
    y_extend=(1, 1000),
    **patch_kw,
):
    if "time" not in da.dims:
        return

    if "diameter" in da.dims:
        dvals = da["diameter"].values
        y0, y1 = dvals[0] - y_extend[0], dvals[-1] + y_extend[1]
    else:
        y0, y1 = y_extend

    collapse_dims = [d for d in da.dims if d not in {"time", "diameter"}]
    d1 = da.sum(dim=collapse_dims) if collapse_dims else da
    mask = ~(d1.sum(dim="diameter") > 0) if "diameter" in d1.dims else ~(d1 > 0)
    mask = np.asarray(mask).ravel()
    if not mask.any():
        return

    ti = np.where(mask)[0]
    split = np.where(np.diff(ti) > 1)[0] + 1
    starts = np.r_[0, split]
    ends = np.r_[split, len(ti)]
    tvals = d1["time"].values

    opts = {
        "facecolor": "lightgray",
        "edgecolor": "black",
        "alpha": 0.2,
        "linewidth": 0.2,
        "hatch": "///",
        "zorder": 1,
        **patch_kw,
    }
    for s, e in zip(starts, ends):
        if e - s < min_consecutive:
            continue
        t0, t1 = tvals[ti[s]], tvals[ti[e - 1]]
        x0, x1 = md.date2num(t0), md.date2num(t1)
        self.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, **opts))


maxes.Axes.format_axis_ticks = format_axis_ticks
maxes.Axes.format_elapsed_time = format_elapsed_time
maxes.Axes.add_missing_data_patches = add_missing_data_patches


In [15]:
def build_common_xlim(
    ds_by_run: dict[str, dict[str, xr.Dataset]],
    *,
    kind: str = "integrated",
    span_min: int = 35,
    anchor: np.datetime64 | None = None,
):
    starts = []
    for run in ds_by_run.values():
        ds = run.get(kind)
        if not isinstance(ds, xr.Dataset):
            continue
        if "time" not in ds.coords or ds.time.size == 0:
            continue
        starts.append(np.datetime64(ds.time.values.min(), "s"))

    if not starts:
        raise ValueError(f"Cannot infer xlim: no datasets with kind='{kind}' and valid time")

    if anchor is None:
        # Domain-specific default: start one minute before flare ignition.
        day = str(min(starts))[:10]
        t0 = np.datetime64(f"{day}T12:29:00")
    else:
        t0 = np.datetime64(anchor, "s")

    return [t0, t0 + np.timedelta64(int(span_min), "m")]



def diagnostics_table(
    ds_by_run: dict[str, dict[str, xr.Dataset]],
    *,
    kind: str = "integrated",
    variable: str = "nf",
    xlim=None,
) -> pd.DataFrame:
    rows = []
    for label, run in ds_by_run.items():
        ds = run.get(kind)
        if ds is None:
            rows.append({"run": label, "status": "missing kind"})
            continue
        if not isinstance(ds, xr.Dataset):
            rows.append({"run": label, "status": f"invalid kind type: {type(ds).__name__}"})
            continue

        row = {
            "run": label,
            "status": "ok",
            "n_cells": int(ds.sizes.get("cell", 1)),
            "n_time": int(ds.sizes.get("time", 0)),
            "time_min": str(ds.time.values.min()).split(".")[0] if "time" in ds.coords and ds.time.size else "-",
            "time_max": str(ds.time.values.max()).split(".")[0] if "time" in ds.coords and ds.time.size else "-",
            "has_var": variable in ds,
        }

        if variable in ds and "time" in ds[variable].dims and xlim is not None:
            in_win = ds[variable].sel(time=slice(xlim[0], xlim[1]))
            row["n_time_in_xlim"] = int(in_win.sizes.get("time", 0))
            row["finite_in_xlim"] = int(np.isfinite(in_win.values).sum())
        rows.append(row)

    return pd.DataFrame(rows)


In [16]:
# ---- user config ----
PROCESSED_ROOT = Path("../../data/processed")
RUNS = [
    # active in figure1
    {"label": "400m, inp 1e6, ccn 0 (run A)", "cs_run": "cs-eriswil__20251129_230943", "exp_id": "20251129231107"},
    # {"label": "400m, inp 1e6, ccn 0 (run B)", "cs_run": "cs-eriswil__20251129_230943", "exp_id": "20260119103733"},

    # previously commented in figure1 (now included)
    # {"label": "400m, inp 1e6, ccn 400 (columnar)", "cs_run": "cs-eriswil__20251125_114053", "exp_id": "20260120122711"},
    # {"label": "400m, inp 1e6, ccn 0 (spherical)", "cs_run": "cs-eriswil__20251125_114053", "exp_id": "20251125114238"},
    # {"label": "400m, inp 1e6, ccn 400 (analytic)", "cs_run": "cs-eriswil__20260127_211338", "exp_id": "20260127211431"},
    # {"label": "400m, inp 1e6, ccn 400 (planar)", "cs_run": "cs-eriswil__20260127_211338", "exp_id": "20260127211551"},
    # {"label": "400m, inp 1e6, ccn 400 (spherical)", "cs_run": "cs-eriswil__20260121_131528", "exp_id": "20260121131550"},
    # {"label": "400m, inp 1e6, ccn 400 (columnar 2)", "cs_run": "cs-eriswil__20260121_131528", "exp_id": "20260121131632"},
    # {"label": "100m, inp 1e6, ccn 0 (columnar)", "cs_run": "cs-eriswil__20260123_180947", "exp_id": "20260123181336"},
    # {"label": "100m, inp 1e6, ccn 400 (columnar)", "cs_run": "cs-eriswil__20260123_180947", "exp_id": "20260123181750"},
    # {"label": "100m, inp 1e6, ccn 400 (spherical, same exp)", "cs_run": "cs-eriswil__20260123_180947", "exp_id": "20260123181750"},
]

KINDS = ("integrated",)

datasets = load_plume_path_runs(
    RUNS,
    processed_root=PROCESSED_ROOT,
    kinds=KINDS,
)

try:
    xlim = build_common_xlim(datasets, kind="integrated", span_min=35)
except ValueError:
    # Fallback window if no valid times are discoverable in current environment.
    xlim = [np.datetime64("2023-01-25T12:29:00"), np.datetime64("2023-01-25T13:04:00")]

diag = diagnostics_table(datasets, kind="integrated", variable="nf", xlim=xlim)
print(diag.to_string(index=False))





                         run status  n_cells  n_time            time_min            time_max  has_var  n_time_in_xlim  finite_in_xlim
400m, inp 1e6, ccn 0 (run A)     ok        6     251 2023-01-25T12:23:20 2023-01-25T13:05:00     True             211           14580


In [20]:
# Waterfall plot
import matplotlib.colors as mcolors
def get_moments(conc, diam):
    """Return weighted mean and variance."""
    tot = np.nansum(conc)
    if tot <= 0:
        return np.nan, np.nan
    mu = np.nansum(conc * diam) / tot
    var = np.nansum(conc * (diam - mu) ** 2) / tot
    return mu, var


def collapse_cell_dim(da):
    """Sum over cell if present."""
    return da.sum("cell") if "cell" in da.dims else da


def make_time_bounds(t0, time_windows, default_start):
    """Return consecutive panel boundaries from t0 and window strings."""
    t_start = np.datetime64(t0) if t0 else default_start
    return [t_start] + [t_start + pd.Timedelta(tw) for tw in time_windows]


def compute_layer_colors(da_w, alt_bands, t_lo, t_hi, cmap, n_bands):
    """Color by mean w in each altitude band when da_w is given."""
    if da_w is None:
        return [cmap(i / n_bands) for i in range(n_bands)]

    w_cmap = plt.cm.coolwarm
    w_norm = plt.Normalize(vmin=-2, vmax=2)
    out = []
    for hi, lo in alt_bands:
        w_slab = da_w.sel(time=slice(t_lo, t_hi), altitude=slice(hi, lo))
        if "cell" in w_slab.dims:
            w_slab = w_slab.mean("cell")
        out.append(w_cmap(w_norm(w_slab.mean().values)))
    return out


def make_phase_styles(color):
    """Return plotting styles for liquid and frozen PSD."""
    c = color[:3]
    return {
        "nw": {"ec": (*c, 0.8), "ls": "--", "lw": 0.85, "fc": (*c, 0.2)},
        "nf": {"ec": (*c, 1.0), "ls": "-", "lw": 0.85, "fc": (*c, 0.6), "hatch": "////"},
    }


def plot_psd_band(ax, da, style, t_lo, t_hi, hi, lo, d_proj, y_off, zlim, log_zlim_lo, idx_faint_start):
    """Draw one PSD slab (faint left segment + solid right segment)."""
    slab = da.sel(time=slice(t_lo, t_hi), altitude=slice(hi, lo)).sum(["time", "altitude"])
    y = np.where(slab > zlim[0], np.log10(slab + 1e-14) - log_zlim_lo + y_off, np.nan)

    faint = dict(style)
    faint.update({"fc": (*style["fc"][:3], 0.05), "ec": (*style["ec"][:3], 0.1), "hatch": None})
    ax.fill_between(d_proj[idx_faint_start:31], y_off, y[idx_faint_start:31], step="mid", **faint)
    ax.fill_between(d_proj[30:], y_off, y[30:], step="mid", **style)


def draw_reference_grid(ax, n_bands, x_shift, y_shift):
    """Draw reference slanted grid in log-diameter space."""
    for d in [1e-3, 1e-2, 1e-1, 1, 10, 100, 1000]:
        xs = [d * 10 ** ((i - n_bands + 1) * x_shift) for i in range(n_bands)]
        ys = [i * y_shift for i in range(n_bands)]
        ax.plot(xs, ys, "k:", lw=0.2, alpha=0.8, zorder=1)


def draw_cloud_top_base(ax, nw, nf, t_lo, t_hi, cloud_thresh, alt_bands, diam, n_bands, x_shift, y_shift, idx_faint_start):
    """Draw CT/CB horizontal markers in projected coordinates."""
    alt_tops = np.array(alt_bands)[:, 0]
    prof = (nw + nf).sel(time=slice(t_lo, t_hi)).mean("time").sum("diameter")
    active = prof.altitude[prof > cloud_thresh]
    if len(active) == 0:
        return

    h0, h1 = alt_bands[0][0], alt_bands[1][0]
    for name, alt in [("CT", active.max().values), ("CB", active.min().values)]:
        idx_x = np.argmin(np.abs(alt - alt_tops))
        x_off = (idx_x - n_bands + 1) * x_shift
        d_proj = diam * (10 ** x_off)
        y_pos = (alt - h0) / (h1 - h0) * y_shift
        ax.plot([d_proj[idx_faint_start], d_proj[-1]], [y_pos, y_pos], c="black", ls="--", lw=0.6, alpha=0.7, zorder=3)
        ax.text(0.99 + x_off * 0.1, y_pos + 0.6, f"{name} ({int(alt)}m)", transform=ax.get_yaxis_transform(), ha="right", va="top", fontsize=7, color="black", alpha=0.8)


def draw_concentration_scale(ax, xlim, zlim, n_bands, y_shift):
    """Draw concentration axis glyph in first column panels."""
    y_front = (n_bands - 1) * y_shift
    sb_x = xlim[0] * 2
    log_min, log_max = np.log10(zlim[0]), np.log10(zlim[1])
    height = log_max - log_min

    ax.plot([sb_x, sb_x], [y_front, y_front + height], "k-", lw=1.5, zorder=100)
    for i in range(0, int(height) + 1, 2):
        y_tick = y_front + i
        val_exp = int(log_min + i)
        ax.plot([sb_x, sb_x * 0.6], [y_tick, y_tick], "k-", lw=1, zorder=100)
        ax.text(sb_x * 0.4, y_tick, f"$10^{{{val_exp}}}$", ha="right", va="center", weight="bold")


def build_stats_rows(nw, nf, diam, t_lo, t_hi):
    """Return table rows for CDNC and ICNC diagnostics."""
    dt_s = (t_hi - t_lo) / np.timedelta64(1, "s")
    rows = []

    for da in [nw, nf]:
        sum_dims = [d for d in ("altitude", "diameter") if d in da.dims]
        sel = {"time": slice(t_lo, t_hi), "diameter": slice(diam[30], None)}

        sub = da.sel(sel).mean("time")
        if "altitude" in sub.dims:
            sub = sub.sum("altitude")

        dsub = da.diff("time").sel(sel).mean("time")
        if "altitude" in dsub.dims:
            dsub = dsub.sum("altitude")

        tot = sub.sum(skipna=True).values
        if tot > 0:
            mu = (sub * diam[30:]).sum(skipna=True).values / tot
            var = (sub.values * (diam[30:] - mu) ** 2).sum() / tot
            std = np.sqrt(var)
        else:
            mu, std = np.nan, np.nan

        ts = da.sum(sum_dims).sel(time=slice(t_lo, t_hi))
        rate = (ts[-1] - ts[0]) / dt_s if len(ts) > 1 and dt_s > 0 else 0
        rate2 = dsub.sum(skipna=True).values if len(dsub) > 1 and dt_s > 0 else 0
        rows.append([f"{mu:.1f}", f"{std:.1f}", f"{rate / 60:.2f}", f"{rate2 / 60:.2f}"])

    return list(map(list, zip(*rows)))


def prepared_to_latex_table(prepared, holimo_obs=None):
    """Return diagnostics table with HOLIMO merged into model columns."""

    def fmt_pair(model_val, holimo_val, nd=1):
        m = f"{model_val:.{nd}f}"
        if holimo_val is None or not np.isfinite(holimo_val):
            return m
        h = f"{holimo_val:.{nd}f}"
        if model_val > holimo_val:
            return rf"\textbf{{{m}}}/{h}"
        if holimo_val > model_val:
            return rf"{m}/\textbf{{{h}}}"
        return f"{m}/{h}"

    t_ref = prepared["bounds"][0]
    rows = []

    for panel in prepared["panel_data"]:
        dt_lo = (panel["t_lo"] - t_ref) / np.timedelta64(1, "m")
        dt_hi = (panel["t_hi"] - t_ref) / np.timedelta64(1, "m")
        s = panel["stats_rows"]

        cdnc_mean = float(s[0][0])
        cdnc_std = float(s[1][0])
        cdnc_growth = float(s[2][0])
        icnc_mean = float(s[0][1])
        icnc_std = float(s[1][1])
        icnc_growth = float(s[2][1])

        label = f"{dt_lo:.2f}-{dt_hi:.2f}"
        holimo_mean = None
        holimo_std = None

        if holimo_obs:
            matches = [obs for obs in holimo_obs if dt_lo <= obs.get("growth_min", np.nan) < dt_hi]
            if matches:
                missions = ",".join(obs["id"] for obs in matches)
                label = f"{label}/{missions}"
                psd = np.concatenate([np.asarray(obs["psd"], dtype=float) for obs in matches])
                diam = np.concatenate([np.asarray(obs["diameter"], dtype=float) for obs in matches])
                holimo_mean, var_h = get_moments(psd, diam)
                holimo_std = np.sqrt(var_h) if np.isfinite(var_h) else None

        rows.append(
            {
                "time_frame_min": label,
                "cdnc_mean_um": f"{cdnc_mean:.1f}",
                "cdnc_std_um": f"{cdnc_std:.1f}",
                "cdnc_growth_rate": f"{cdnc_growth:.2f}",
                "icnc_mean_um": fmt_pair(icnc_mean, holimo_mean, nd=1),
                "icnc_std_um": fmt_pair(icnc_std, holimo_std, nd=1),
                "icnc_growth_rate": f"{icnc_growth:.2f}",
            }
        )

    df = pd.DataFrame(rows)
    latex = df.to_latex(index=False, escape=False)
    return df, latex


def save_latex_table(prepared, out_file, holimo_obs=None):
    """Write diagnostics table to .tex file for \input in LaTeX docs."""
    _, latex = prepared_to_latex_table(prepared, holimo_obs=holimo_obs)
    out_path = Path(out_file)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(latex, encoding="utf-8")
    return out_path


def add_stats_table(ax, stats_rows):
    """Render stats table into one panel."""
    labels = [
        r"$\mu$ ($\mu$m)",
        r"$\sigma$ ($\mu$m)",
        r"$\dot{N}$ (L$^{-1}$s$^{-1}$)",
        r"$\dot{N}$ (L$^{-1}$min$^{-1}$)",
    ]
    table_content = [r + [l] for r, l in zip(stats_rows, labels)]
    tbl = ax.table(cellText=table_content, colLabels=["CDNC", "ICNC", ""], loc="upper right", bbox=[0.7, 0.75, 0.25, 0.22])
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(6)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_linewidth(0)
        cell.set_text_props(ha="left" if c == 2 else "right")


def format_waterfall_panel(ax, idx, t_lo, t_hi, t_start, xlim):
    """Apply shared axis formatting for each panel."""
    dt_lo = (t_lo - t_start) / np.timedelta64(1, "m")
    dt_hi = (t_hi - t_start) / np.timedelta64(1, "m")
    panel_label = f"({chr(65 + idx)}) {dt_lo:.1f} — {dt_hi:.1f} min"

    ax.set(xscale="log", xlim=xlim, yticks=[])
    for spine in ["left", "top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, p: f"{x:g}"))
    ax.text(
        0.02,
        0.96,
        panel_label,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=9,
        fontweight="bold",
        bbox={"facecolor": "white", "edgecolor": "0.25", "alpha": 0.9, "boxstyle": "round,pad=0.2"},
    )


def add_altitude_colorbar(fig, axes, cmap, n_bands, alt_bands):
    """Add discrete altitude colorbar with ticks at band boundaries."""
    colors = [cmap(i / n_bands) for i in range(n_bands)]
    cb_bounds = [b[1] for b in alt_bands[::-1]] + [alt_bands[0][0]]
    norm = mcolors.BoundaryNorm(cb_bounds, n_bands)
    sm = plt.cm.ScalarMappable(cmap=mcolors.ListedColormap(colors[::-1]), norm=norm)
    cb = fig.colorbar(sm, ax=axes, pad=0.02, shrink=0.6, aspect=40, label="Altitude / m")
    cb.set_ticks(cb_bounds)
    cb.set_ticklabels([f"{int(l)}" for l in cb_bounds])


def build_holimo_obs_series(ds_hd, time_frames_plume, obs_ids, growth_times_min, var_candidates=None):
    """Build observed ice PSD sums tied to plume-growth timestamps."""
    if var_candidates is None:
        var_candidates = ["Ice_PSDnoNorm", "Ice_PSDMnoNorm", "Ice_PSDlinNorm", "Ice_PSDlogNorm"]

    obs_var = next((v for v in var_candidates if v in ds_hd.data_vars), None)
    if obs_var is None or ds_hd.time.size < 2:
        return []

    dt_s = float((ds_hd.time[1] - ds_hd.time[0]) / np.timedelta64(1, "s"))
    if not np.isfinite(dt_s) or dt_s <= 0:
        return []

    out = []
    for obs_id, growth_min, (t0, t1) in zip(obs_ids, growth_times_min, time_frames_plume):
        obs = (ds_hd[obs_var] * dt_s).sel(time=slice(t0, t1)).sum("time")
        unit_factor = 1e3 if 'cm-3' in obs.attrs['unit'] else 1e0
        unit = obs.attrs['unit'].replace('cm-3', 'L-1') if 'cm-3' in obs.attrs['unit'] else obs.attrs['unit']
        out.append(
            {
                "id": obs_id,
                "growth_min": float(growth_min),
                "diameter": obs.diameter.values,
                "psd": np.asarray(obs.values, dtype=float) * unit_factor,
                "unit": unit,
            }
        )
    return out


def overlay_holimo_obs(ax, holimo_obs, t_lo, t_hi, t_start, zlim, log_zlim_lo):
    """Overlay observed PSD in panel matching plume growth-time window."""
    dt_lo = (t_lo - t_start) / np.timedelta64(1, "m")
    dt_hi = (t_hi - t_start) / np.timedelta64(1, "m")

    # Reuse model-like phase styling but keep HOLIMO magenta.
    magenta = (1.0, 0.0, 1.0, 1.0)
    obs_style = make_phase_styles(magenta)["nf"]

    for obs in holimo_obs:
        g = obs["growth_min"]
        if not (dt_lo <= g < dt_hi):
            continue

        y = np.where(obs["psd"] > zlim[0], np.log10(obs["psd"] + 1e-14) - log_zlim_lo, np.nan)

        faint = dict(obs_style)
        faint.update({"fc": (*obs_style["fc"][:3], 0.08), "ec": (*obs_style["ec"][:3], 0.15), "hatch": None})
        n = len(obs["diameter"])
        i_split = max(2, n // 3)

        ax.fill_between(obs["diameter"][:i_split], 0, y[:i_split], step="mid", **faint, zorder=120)
        ax.fill_between(obs["diameter"][i_split - 1 :], 0, y[i_split - 1 :], step="mid", **obs_style, zorder=121)

        ax.plot(obs["diameter"], y, c="magenta", lw=1.0, ls="-", drawstyle="steps-mid", zorder=122)
        ax.text(0.02, 0.86, f"HOLIMO {obs['id']} (gt={g:.1f} min)", transform=ax.transAxes, ha="left", va="top", fontsize=7.5, color="magenta", weight="bold")


def prepare_psd_waterfall_data(
    da_nw,
    da_nf,
    time_windows,
    t0,
    alt_bands,
    da_w=None,
    cmap=plt.cm.jet,
    x_shift=-0.1,
):
    """Precompute all expensive slices/sums once for fast re-plotting."""
    nf, nw = (collapse_cell_dim(da) for da in [da_nf, da_nw])
    bounds = make_time_bounds(t0, time_windows, nf.time.values[0])

    has_altitude = "altitude" in nf.dims and "altitude" in nw.dims
    alt_bands_used = alt_bands if has_altitude else [(0.0, 0.0)]
    n_bands = len(alt_bands_used)

    diam = nf.diameter.values
    idx_faint_start = np.argmin(np.abs(0.2 - diam))
    delta_t = (nf.time[1] - nf.time[0]).values / np.timedelta64(1, "s")

    panel_data = []
    for idx in range(len(bounds) - 1):
        t_lo, t_hi = bounds[idx], bounds[idx + 1]
        layer_colors = compute_layer_colors(da_w, alt_bands_used, t_lo, t_hi, cmap, n_bands) if has_altitude else [cmap(0.6)]

        bands = []
        for i, ((hi, lo), layer_color) in enumerate(zip(alt_bands_used, layer_colors)):
            x_off = (i - n_bands + 1) * x_shift
            d_proj = diam * (10 ** x_off)

            if has_altitude:
                nw_slab = (nw * delta_t).sel(time=slice(t_lo, t_hi), altitude=slice(hi, lo))
                nf_slab = (nf * delta_t).sel(time=slice(t_lo, t_hi), altitude=slice(hi, lo))
                n_alt_band = max(1, int(nw_slab.altitude.size))
                nw_slab = nw_slab.sum(["time", "altitude"]) / n_alt_band
                nf_slab = nf_slab.sum(["time", "altitude"]) / n_alt_band
            else:
                nw_slab = (nw * delta_t).sel(time=slice(t_lo, t_hi)).sum("time")
                nf_slab = (nf * delta_t).sel(time=slice(t_lo, t_hi)).sum("time")

            bands.append({"idx": i, "hi": hi, "lo": lo, "color": layer_color, "d_proj": d_proj, "nw_slab": nw_slab.values, "nf_slab": nf_slab.values})

        stats_rows = build_stats_rows(nw, nf, diam, t_lo, t_hi)
        panel_data.append({"idx": idx, "t_lo": t_lo, "t_hi": t_hi, "bands": bands, "stats_rows": stats_rows})

    return {
        "nf": nf,
        "nw": nw,
        "bounds": bounds,
        "diam": diam,
        "n_bands": n_bands,
        "idx_faint_start": idx_faint_start,
        "panel_data": panel_data,
        "has_altitude": has_altitude,
        "alt_bands_used": alt_bands_used,
    }


def plot_psd_waterfall(
    prepared,
    alt_bands,
    zlim=(1e0, 1e6),
    xlim=(1e-3, 3e3),
    cmap=plt.cm.jet,
    n_cols=3,
    y_shift=-0.75,
    x_shift=-0.1,
    cloud_thresh=1e0,
    holimo_obs=None,
    show_cloud_bounds=True,
    show_stats_table=True,
    show_small_diameters=True,
):
    """3D-effect Waterfall Plot for liquid/frozen PSD fields using precomputed data."""
    nf = prepared["nf"]
    nw = prepared["nw"]
    bounds = prepared["bounds"]
    diam = prepared["diam"]
    n_bands = prepared["n_bands"]
    idx_faint_start = prepared["idx_faint_start"]
    panel_data = prepared["panel_data"]
    has_altitude = prepared.get("has_altitude", True)
    alt_bands_used = prepared.get("alt_bands_used", alt_bands)
    log_zlim_lo = np.log10(zlim[0])

    n_panels = len(panel_data)
    n_rows = -(-n_panels // n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4.5, n_rows * 2.8), layout="constrained")

    for idx, ax in enumerate(np.atleast_1d(axes).flat):
        if idx >= n_panels:
            ax.set_visible(False)
            continue

        p = panel_data[idx]
        t_lo, t_hi = p["t_lo"], p["t_hi"]

        for band in p["bands"]:
            i = band["idx"]
            y_off = i * y_shift
            styles = make_phase_styles(band["color"])

            y_nw = np.where(band["nw_slab"] > zlim[0], np.log10(band["nw_slab"] + 1e-14) - log_zlim_lo + y_off, np.nan)
            y_nf = np.where(band["nf_slab"] > zlim[0], np.log10(band["nf_slab"] + 1e-14) - log_zlim_lo + y_off, np.nan)

            if show_small_diameters:
                ax.fill_between(band["d_proj"], y_off, y_nw, step="mid", **styles["nw"])
            else:
                faint_nw = dict(styles["nw"])
                faint_nw.update({"fc": (*styles["nw"]["fc"][:3], 0.05), "ec": (*styles["nw"]["ec"][:3], 0.1), "hatch": None})
                ax.fill_between(band["d_proj"][idx_faint_start:31], y_off, y_nw[idx_faint_start:31], step="mid", **faint_nw)
                ax.fill_between(band["d_proj"][30:], y_off, y_nw[30:], step="mid", **styles["nw"])

            ax.fill_between(band["d_proj"], y_off, y_nf, step="mid", **styles["nf"])

        if holimo_obs:
            overlay_holimo_obs(ax, holimo_obs, t_lo, t_hi, bounds[0], zlim, log_zlim_lo)

        draw_reference_grid(ax, n_bands, x_shift, y_shift)
        if show_cloud_bounds and has_altitude and n_bands > 1:
            draw_cloud_top_base(ax, nw, nf, t_lo, t_hi, cloud_thresh, alt_bands_used, diam, n_bands, x_shift, y_shift, idx_faint_start)

        if idx % n_cols == 0:
            draw_concentration_scale(ax, xlim, zlim, n_bands, y_shift)

        if show_stats_table:
            add_stats_table(ax, p["stats_rows"])
        format_waterfall_panel(ax, idx, t_lo, t_hi, bounds[0], xlim)

    if has_altitude:
        add_altitude_colorbar(fig, axes, cmap, n_bands, alt_bands_used)
    fig.supylabel(r"Number Concentration (at lowest altitude) / L$^{-1}$", fontsize=12)
    fig.supxlabel(r"Diameter / $\mu$m", fontsize=12)
    return fig, axes


def prepare_waterfall_inputs(ds_vertical):
    """Prepare NF/NW arrays and harmonize NW units."""
    plot_nf = xr.where(ds_vertical["nf"] > 0, ds_vertical["nf"], np.nan)
    plot_nw = xr.where(ds_vertical["nw"] > 0, ds_vertical["nw"], np.nan) * 1e3
    plot_nw.attrs["units"] = "L⁻¹"
    return plot_nw, plot_nf


def render_waterfall_case(
    cs_idx,
    cs_run_datasets,
    cfg,
    alt_bands,
    cmap,
    time_windows,
    zlim,
    xlim,
    y_shift=-0.5,
    x_shift=-0.1,
    holimo_obs=None,
    show_cloud_bounds=True,
    show_stats_table=True,
):
    """Prepare once, then plot from precomputed structures."""
    print(f"\nselected cs_idx: {cs_idx}")

    run_ds = cs_run_datasets[cs_idx]
    ds_case = run_ds.get("vertical", run_ds.get("integrated"))
    if ds_case is None:
        raise KeyError(f"No 'vertical' or 'integrated' dataset found for '{cs_idx}'")

    if "cell" in ds_case.dims:
        ds_case = ds_case.sum("cell")

    plot_nw, plot_nf = prepare_waterfall_inputs(ds_case)

    t0_cfg = cfg.get(cs_idx, {}).get("flare_start_datetime") if isinstance(cfg, dict) else None
    t0 = np.datetime64(t0_cfg) if t0_cfg else np.datetime64(ds_case.time.values.min(), "s")

    prepared = prepare_psd_waterfall_data(
        plot_nw,
        plot_nf,
        time_windows=time_windows,
        t0=t0,
        alt_bands=alt_bands,
        da_w=None,
        cmap=cmap,
        x_shift=x_shift,
    )

    return plot_psd_waterfall(
        prepared,
        alt_bands=alt_bands,
        zlim=zlim,
        xlim=xlim,
        cmap=cmap,
        y_shift=y_shift,
        x_shift=x_shift,
        holimo_obs=holimo_obs,
        show_cloud_bounds=show_cloud_bounds,
        show_stats_table=show_stats_table,
    )




In [18]:
# Load HOLIMO data

holimo_file      = '../../data/observations/holimo_data/CL_20230125_1000_1140_SM058_SM060_ts1.nc'

time_window_holimo = (np.datetime64('2023-01-25T10:10:00'), np.datetime64('2023-01-25T12:00:00'))
time_frames_plume = [   
                    [ np.datetime64('2023-01-25T10:55:00'), np.datetime64('2023-01-25T11:10:00')   ],
                    [ np.datetime64('2023-01-25T10:35:00'), np.datetime64('2023-01-25T10:50:00')   ],
                    [ np.datetime64('2023-01-25T11:24:00'), np.datetime64('2023-01-25T11:39:00')   ],
                    ]

obs_ids = [
    "SM059", 
    "SM058", 
    "SM060"
    ]
growth_times_min = [
    6.1, 
    8.0, 
    9.1,
    ]
seeding_start_times = [
    np.datetime64('2023-01-25T10:50:00'),
    np.datetime64('2023-01-25T10:28:00'),
    np.datetime64('2023-01-25T11:15:00'),
]




ds_holimo, lbb, cbb = load_and_prepare_holimo(holimo_file)
ds_holimo = ds_holimo.sel(time=slice(*time_window_holimo))
ds_holimo = ds_holimo.assign_coords({'diameter': ds_holimo.diameter * 1e6})
ds_hd10   = ds_holimo.resample(time='10s').mean()


# Optional HOLIMO overlay: requires ds_hd10, obs_ids, time_frames_plume, growth_times_min.
holimo_ice_var = "Ice_Pristine_PSDnoNorm"
holimo_overlay_cfg = None
if all(name in globals() for name in ["ds_hd10", "obs_ids", "time_frames_plume", "growth_times_min"]):
    holimo_overlay_cfg = {
        "ds_hd10": ds_hd10,
        "obs_ids": obs_ids,
        "time_frames_plume": time_frames_plume,
        "growth_times_min": growth_times_min,
        "seeding_start_times": seeding_start_times,
        "var": holimo_ice_var,
        "threshold": 1e-10,
        "unit_factor": 1e3,
        "scatter_cmap": new_jet3,
        "markers": ["o", "s", "^"],
        "sizes": [16, 18, 20],
        "alpha": 0.9,
        "edgecolor": "black",
        "linewidth": 0.35,
        "legend_loc": "upper right",
    }


# HOLIMO observational PSDs in requested order: SM059, SM058, SM060
holimo_obs = build_holimo_obs_series(ds_hd10, time_frames_plume, obs_ids, growth_times_min, var_candidates=["Ice_PSDnoNorm"])


In [ ]:

# ##########################################
# MAIN PLOT CALL
# Example usage for figure 13

# Load Model PSDs
cs_run_datasets = load_plume_path_runs(
    RUNS,
    processed_root=PROCESSED_ROOT,
    kinds=("vertical", "integrated"),
)

# Print diagnostics table
time_frame = [np.datetime64("2023-01-25T12:30:00"), np.datetime64("2023-01-25T13:04:00")]
diag = diagnostics_table(datasets, kind="integrated", variable="nf", xlim=time_frame)
print(diag.to_string(index=False))


cfg = { run["label"]: {"flare_start_datetime": run.get("flare_start_datetime")} for run in globals().get("RUNS", []) if "label" in run}

cs_indices = list(cs_run_datasets.keys())
print(f"list of cs_indices:\n", "\n ".join(cs_indices))

idx_dbg = 1

alt_bands = [(1350, 1275), (1275, 1200), (1200, 1125), (1125, 1050), (1050, 975), (975, 900), (900, 850), (850, 800)]
time_windows = ["0.25min", "0.5min", "1min", "2.5min", "5min", "7.5min", "10min", "15min", "25min"]
zlim = (1e1, 1e8)
xlim = (1e-3, 9e3)
cmap = make_pastel(new_jet3, desaturation=0.25, darken=0.90)
show_cloud_bounds = True
show_stats_table = False
show_suptitle = False
show_small_diameters=False
prepped_list = []
for cs_idx in cs_indices[:idx_dbg]:
    print(f"Processing cs_idx: {cs_idx}")
    # Prepare once (expensive), then reuse for plotting if figure style is adjusted.
    run_ds = cs_run_datasets[cs_idx]
    ds_case = run_ds.get("vertical", run_ds.get("integrated"))
    if ds_case is None:
        continue

    if "cell" in ds_case.dims:
        ds_case = ds_case.sum("cell")

    plot_nw, plot_nf = prepare_waterfall_inputs(ds_case)

    prepared = prepare_psd_waterfall_data(
        plot_nw,
        plot_nf,
        time_windows=time_windows,
        t0=np.datetime64("2023-01-25T12:30:00"),
        alt_bands=alt_bands,
        da_w=None,
        cmap=cmap,
        x_shift=-0.1,
    )
    fig_wf_alt, axes_wf_alt = plot_psd_waterfall(
        prepared,
        alt_bands=alt_bands,
        zlim=zlim,
        xlim=xlim,
        cmap=cmap,
        y_shift=-0.5,
        x_shift=-0.1,
        show_cloud_bounds=show_cloud_bounds,
        show_stats_table=show_stats_table,
        # holimo_obs=holimo_obs,
    )

    if show_suptitle:
        fig_wf_alt.suptitle("CDNC and ICNC PSD waterfall by altitude and time after seeding", fontsize=13, weight="semibold")
    
    run_id = run_ds.attrs['run_id']
    figure_file = f"output/figure13_psd_waterfall_alt_time_{run_id}.png"
    fig_wf_alt.savefig(figure_file, bbox_inches="tight", dpi=300)

    holimo_for_table = holimo_obs if "holimo_obs" in globals() else None
    stats_df, stats_latex = prepared_to_latex_table(prepared, holimo_obs=holimo_for_table)
    tex_file = save_latex_table(
        prepared,
        f"output/tables/figure13_psd_stats_{run_id}.tex",
        holimo_obs=holimo_for_table,
    )
    print(f"\nLaTeX diagnostics table for {run_id}:\n")
    # print(stats_latex)
    print(stats_df.to_string(index=False))
    print(f"\n\nLaTeX saved table: {tex_file}")


                         run status  n_cells  n_time            time_min            time_max  has_var  n_time_in_xlim  finite_in_xlim
400m, inp 1e6, ccn 0 (run A)     ok        6     251 2023-01-25T12:23:20 2023-01-25T13:05:00     True             205           14350
list of cs_indices:
 400m, inp 1e6, ccn 0 (run A)
Processing cs_idx: 400m, inp 1e6, ccn 0 (run A)


## Figure caption

**Figure 13.** Altitude-resolved PSD waterfall plot for plume-path evolution after seeding. For successive post-ignition time windows, ridgelines show number concentration by diameter within stacked altitude bands, visualized in a skewed waterfall coordinate system to preserve vertical structure and temporal progression in one panel. Liquid-phase concentrations (CDNC) and frozen-phase concentrations (ICNC) are overlaid to highlight phase partitioning and the transition from droplet-dominated to ice-dominated spectra.



```latex

\begin{tabular}{lllllll}
    \toprule
    time_frame_min & cdnc_mean_um & cdnc_std_um & cdnc_growth_rate & icnc_mean_um & icnc_std_um & icnc_growth_rate \\
    \midrule
    0.00-0.25 & 6.0 & 3.6 & 27.31 & 10.9 & 5.3 & 0.55 \\
    0.25-0.50 & 6.0 & 3.6 & -9088.40 & 15.6 & 5.5 & 0.42 \\
    0.50-1.00 & 6.1 & 3.6 & -5.24 & 19.5 & 7.2 & 0.48 \\
    1.00-2.50 &6.1 & 3.6 & -34.20 & 25.1 & 10.6 & -0.10 \\
    2.50-5.00 & 6.3 & 3.6 & 930.28 & 44.3 & 13.7 & -0.08 \\
    5.00-7.50 & 6.2 & 3.5 & -19.33 & 63.3 & 14.2 & -0.04 \\
    7.50-10.00 & 6.3 & 3.6 & 35.20 & 81.7 & 15.5 & -0.02 \\
    10.00-15.00 & 6.1 & 3.4 & -480.08 & 102.8 & 21.6 & -0.01 \\
    15.00-25.00 & 5.9 & 3.2 & -17.53 & 149.3 & 25.2 & 0.01 \\
    \bottomrule
\end{tabular}


